# 08 — Build the PolicyPal Vector Search index

This notebook creates a Delta Sync Vector Search index over
`${catalog}.pawshield.policy_chunks`, then runs the canonical
Sarah Chen retrieval question two ways — with and without a metadata
filter — to show why the filter belongs at the index, not in
application code.

**Prerequisites**:

1. The `policy_chunks` table exists with
   Change Data Feed enabled.
2. A Vector Search endpoint has been provisioned in the workspace.
   Set the `vs_endpoint` widget to its name. Endpoint creation is a
   workspace-admin action (`Compute → Vector Search → Create endpoint`)
   and is *not* run from this notebook because endpoint provisioning
   is a workspace-level decision with cost implications.
3. The `databricks-gte-large-en` foundation-model embedding endpoint is
   available in the workspace (it is the default pay-per-token
   embedding model on Databricks).

In [0]:
dbutils.widgets.text("catalog", "genaicert")
dbutils.widgets.text("vs_endpoint", "pawshield-vs")
dbutils.widgets.text("index_short_name", "policy_chunks_index")
dbutils.widgets.text("embedding_endpoint", "databricks-gte-large-en")

catalog = dbutils.widgets.get("catalog")
vs_endpoint = dbutils.widgets.get("vs_endpoint")
index_short = dbutils.widgets.get("index_short_name")
embedding_endpoint = dbutils.widgets.get("embedding_endpoint")

source_table = f"{catalog}.pawshield.policy_chunks"
index_name = f"{catalog}.pawshield.{index_short}"

print(f"catalog:            {catalog}")
print(f"vs_endpoint:        {vs_endpoint}")
print(f"index_short_name:   {index_short}")
print(f"embedding_endpoint: {embedding_endpoint}")
print(f"source_table:       {source_table}")
print(f"index_name:         {index_name}")

catalog:            genaicert
vs_endpoint:        pawshield-vs
index_short_name:   policy_chunks_index
embedding_endpoint: databricks-gte-large-en
source_table:       genaicert.pawshield.policy_chunks
index_name:         genaicert.pawshield.policy_chunks_index


## 1. Install the Vector Search client

In [0]:
%pip install --quiet "databricks-vectorsearch<0.74"

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
# Re-read widget values after Python restart.
catalog = dbutils.widgets.get("catalog")
vs_endpoint = dbutils.widgets.get("vs_endpoint")
index_short = dbutils.widgets.get("index_short_name")
embedding_endpoint = dbutils.widgets.get("embedding_endpoint")
source_table = f"{catalog}.pawshield.policy_chunks"
index_name = f"{catalog}.pawshield.{index_short}"

## 2. Confirm the source table is index-ready

Two checks:

* `policy_chunks` exists and has rows.
* Change Data Feed is enabled (required for Delta Sync indexes;
  the index syncs incrementally off CDF, not by re-scanning).

In [0]:
%sql
SELECT count(*) AS chunk_count FROM IDENTIFIER(:catalog || '.pawshield.policy_chunks');

chunk_count
1140


In [0]:
%sql
SHOW TBLPROPERTIES IDENTIFIER(:catalog || '.pawshield.policy_chunks');

key,value
delta.enableChangeDataFeed,true
delta.enableDeletionVectors,true
delta.feature.appendOnly,supported
delta.feature.changeDataFeed,supported
delta.feature.deletionVectors,supported
delta.feature.invariants,supported
delta.minReaderVersion,3
delta.minWriterVersion,7
delta.parquet.compression.codec,zstd


## 3. Confirm the Vector Search endpoint exists

If this raises, the workspace admin needs to create the endpoint
via `Compute → Vector Search → Create endpoint` and re-run the
notebook.

In [0]:
# Source: https://docs.databricks.com/aws/en/generative-ai/create-query-vector-search
# `VectorSearchClient.get_endpoint(name=<endpoint>)` returns the endpoint
# descriptor; querying it confirms the endpoint exists and reports its state.
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient(disable_notice=True)

endpoint = vsc.get_endpoint(name=vs_endpoint)
print(f"Endpoint state: {endpoint.get('endpoint_status', {}).get('state', endpoint)}")

Endpoint state: ONLINE


## 4. Create the Delta Sync index

* `pipeline_type=TRIGGERED` re-syncs on a schedule (or manual call),
  appropriate for policy documents that change roughly twice a year.
  `CONTINUOUS` would pay for change-detection that the source data
  does not need.
* `embedding_source_column="chunk_text"` tells Vector Search to embed
  the text on the platform side using the managed embedding endpoint.
  We do not embed in-notebook.
* `tier` and `state` carry through as filterable metadata columns —
  any column on the source table is automatically queryable at
  retrieval time.

In [0]:
# Source: https://api-docs.databricks.com/python/vector-search/latest/
# `VectorSearchClient.list_indexes(name=<endpoint>)` — the kwarg is `name`,
# verified against `databricks-vectorsearch` (May 2026). We keep the try/except
# only as a safety net for older SDK builds that accepted `endpoint_name=`.
try:
    listed = vsc.list_indexes(name=vs_endpoint)
except TypeError:
    listed = vsc.list_indexes(endpoint_name=vs_endpoint)
existing = [i.get("name") for i in listed.get("vector_indexes", [])]
print(f"Existing indexes on {vs_endpoint}: {existing}")

Existing indexes on pawshield-vs: ['genaicert.pawshield.policy_chunks_index']


In [0]:
if index_name in existing:
    print(f"Index {index_name} already exists; skipping creation.")
else:
    # Source: https://docs.databricks.com/aws/en/generative-ai/create-query-vector-search
    # Managed-embeddings Delta Sync index: pass `embedding_source_column` +
    # `embedding_model_endpoint_name` (mutually exclusive with the
    # self-managed pair `embedding_vector_column` + `embedding_dimension`).
    # `pipeline_type` accepts "TRIGGERED" or "CONTINUOUS" (uppercase per doc
    # examples). `primary_key` must uniquely identify rows (PolicyPal uses a
    # STRING `chunk_id`); embedding source column must be STRING. Source table
    # requires CDF enabled (Change Data Feed on the source table).
    vsc.create_delta_sync_index(
        endpoint_name=vs_endpoint,
        source_table_name=source_table,
        index_name=index_name,
        pipeline_type="TRIGGERED",
        primary_key="chunk_id",
        embedding_source_column="chunk_text",
        embedding_model_endpoint_name=embedding_endpoint,
    )
    print(f"Created Delta Sync index {index_name}")

Index genaicert.pawshield.policy_chunks_index already exists; skipping creation.


## 5. Wait for the initial sync

Initial embed of ~1,140 chunks completes in roughly 2–4 minutes on
the standard endpoint sizing. This cell polls until the index is
`ONLINE` and reports the chunk count.

In [0]:
import time

# Source: https://docs.databricks.com/aws/en/generative-ai/create-query-vector-search
# `VectorSearchClient.get_index(endpoint_name=..., index_name=...)` returns an
# index handle whose `.describe()` reports sync status. The dict shape exposes
# `status.ready` (bool, the gate to serve queries) and `status.detailed_state`
# (e.g., "ONLINE_NO_PENDING_UPDATE", for observability while sync is in flight).
index = vsc.get_index(endpoint_name=vs_endpoint, index_name=index_name)

for _ in range(60):  # up to ~10 minutes
    status = index.describe().get("status", {})
    state = status.get("detailed_state") or status.get("ready") or status
    ready = status.get("ready", False)
    print(f"  state={state} ready={ready}")
    if ready:
        break
    time.sleep(10)

print(index.describe())

  state=ONLINE_NO_PENDING_UPDATE ready=True
{'name': 'genaicert.pawshield.policy_chunks_index', 'endpoint_name': 'pawshield-vs', 'primary_key': 'chunk_id', 'index_type': 'DELTA_SYNC', 'delta_sync_index_spec': {'source_table': 'genaicert.pawshield.policy_chunks', 'embedding_source_columns': [{'name': 'chunk_text', 'embedding_model_endpoint_name': 'databricks-gte-large-en'}], 'pipeline_type': 'TRIGGERED', 'pipeline_id': '017b1d00-8335-4383-a2b8-33b5afa65a72'}, 'status': {'detailed_state': 'ONLINE_NO_PENDING_UPDATE', 'message': 'Index creation succeeded. Check latest status: https://<workspace>.cloud.databricks.com/explore/data/genaicert/pawshield/policy_chunks_index', 'indexed_row_count': 1140, 'triggered_update_status': {'last_processed_commit_version': 9, 'last_processed_commit_timestamp': '2026-05-25T12:47:28Z'}, 'ready': True, 'index_url': '<workspace>.cloud.databricks.com/api/2.0/vector-search/indexes/genaicert.pawshield.policy_chunks_index'}, 'creator': '<user>@<org>.com', 'endpoin

## 6. Query A — Sarah's question with NO metadata filter

PolicyPal's RAG chain will call this index. To
see why the metadata filter is mandatory, run Sarah's question
without it first. Sarah is in TX on PawPlus; her policy is
`pp_plus_tx_v3`. The TX, CA, NY, FL... PawPlus policies all share
Section 4.7's chronic-condition language nearly verbatim (the
state-specific differences live in Section 5). Pure semantic
similarity has no reason to prefer Sarah's own state.

In [0]:
query = "Do chronic conditions still hit the deductible on each visit?"

# Source: https://docs.databricks.com/aws/en/vector-search/query-vector-search
# `index.similarity_search(query_text=..., columns=[...], num_results=k)` is
# the default ANN path (equivalent to `query_type="ann"`). Response envelope
# is `{"manifest": {"columns": [...]}, "result": {"data_array": [[...]], ...}}`
# — this `{manifest, result.data_array}` envelope is the recurring unwrap idiom.
unfiltered = index.similarity_search(
    query_text=query,
    columns=["chunk_id", "doc_id", "tier", "state", "section"],
    num_results=4,
)

import pandas as pd
print("Top-4 unfiltered:")
_unfiltered_df = pd.DataFrame(
    unfiltered["result"]["data_array"],
    columns=[c["name"] for c in unfiltered["manifest"]["columns"]],
)
# `display(...)` renders interactively in the workspace UI but is not
# captured in the static .ipynb output stream. Print the DataFrame as
# text so the contamination demo is visible in exported notebooks too.
display(_unfiltered_df)
print(_unfiltered_df.to_string())

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
Top-4 unfiltered:


chunk_id,doc_id,tier,state,section,score
pp_plus_il_v3__chunk_018,pp_plus_il_v3,PawPlus,IL,4.7 Chronic condition exclusions and reimbursement handling,0.6461312020218786
pp_plus_tx_v3__chunk_018,pp_plus_tx_v3,PawPlus,TX,4.7 Chronic condition exclusions and reimbursement handling,0.6461312020218786
pp_plus_ca_v3__chunk_018,pp_plus_ca_v3,PawPlus,CA,4.7 Chronic condition exclusions and reimbursement handling,0.6461312020218786
pp_premier_fl_v3__chunk_018,pp_premier_fl_v3,FureverPremier,FL,4.7 Chronic condition exclusions and reimbursement handling,0.6460916815196058


                      chunk_id            doc_id            tier state                                                      section     score
0     pp_plus_il_v3__chunk_018     pp_plus_il_v3         PawPlus    IL  4.7 Chronic condition exclusions and reimbursement handling  0.646131
1     pp_plus_tx_v3__chunk_018     pp_plus_tx_v3         PawPlus    TX  4.7 Chronic condition exclusions and reimbursement handling  0.646131
2     pp_plus_ca_v3__chunk_018     pp_plus_ca_v3         PawPlus    CA  4.7 Chronic condition exclusions and reimbursement handling  0.646131
3  pp_premier_fl_v3__chunk_018  pp_premier_fl_v3  FureverPremier    FL  4.7 Chronic condition exclusions and reimbursement handling  0.646092


## 7. Query B — same question, filtered to Sarah's policy

On a standard endpoint the filter typically scopes candidates to
Sarah's own state and tier before ranking — but this pre-filter
ordering is not a formally documented contract, so treat
it as the typical effect, not a guarantee. In practice the top-1
chunk should now consistently be `pp_plus_tx_v3 — 4.7 Chronic
condition exclusions and reimbursement handling`.

In [0]:
# Source: https://docs.databricks.com/aws/en/vector-search/query-vector-search
# Standard endpoints take filters as a Python dict (`{"col": "val"}`, with
# operator-suffixed keys like `{"id >": 200}`, `{"col LIKE": "x"}`, `{"col NOT": "y"}`).
# Storage-optimized endpoints use a SQL string instead — the two are NOT
# interchangeable. PolicyPal is on a standard endpoint, so the dict form is
# the documented call shape.
filtered = index.similarity_search(
    query_text=query,
    columns=["chunk_id", "doc_id", "tier", "state", "section"],
    num_results=4,
    filters={"state": "TX", "tier": "PawPlus"},
)

print("Top-4 filtered (state=TX, tier=PawPlus):")
_filtered_df = pd.DataFrame(
    filtered["result"]["data_array"],
    columns=[c["name"] for c in filtered["manifest"]["columns"]],
)
# Mirror the unfiltered display+print pattern so the contrast between
# Query A and Query B is captured in static .ipynb output.
display(_filtered_df)
print(_filtered_df.to_string())

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
Top-4 filtered (state=TX, tier=PawPlus):


chunk_id,doc_id,tier,state,section,score
pp_plus_tx_v3__chunk_018,pp_plus_tx_v3,PawPlus,TX,4.7 Chronic condition exclusions and reimbursement handling,0.646131231056491
pp_plus_tx_v3__chunk_009,pp_plus_tx_v3,PawPlus,TX,4.2 Pre-existing condition exclusion,0.5589479381938165
pp_plus_tx_v3__chunk_013,pp_plus_tx_v3,PawPlus,TX,4.2 Pre-existing condition exclusion,0.5589479381938165
pp_plus_tx_v3__chunk_000,pp_plus_tx_v3,PawPlus,TX,front-matter,0.5582577749835014


                   chunk_id         doc_id     tier state                                                      section     score
0  pp_plus_tx_v3__chunk_018  pp_plus_tx_v3  PawPlus    TX  4.7 Chronic condition exclusions and reimbursement handling  0.646131
1  pp_plus_tx_v3__chunk_009  pp_plus_tx_v3  PawPlus    TX                         4.2 Pre-existing condition exclusion  0.558948
2  pp_plus_tx_v3__chunk_013  pp_plus_tx_v3  PawPlus    TX                         4.2 Pre-existing condition exclusion  0.558948
3  pp_plus_tx_v3__chunk_000  pp_plus_tx_v3  PawPlus    TX                                                 front-matter  0.558258


## 8. Hybrid search — combine vector similarity with keyword scoring

When the query contains literal terms that *must* appear in the
matching chunk (e.g., section names, regulatory phrases,
product-specific jargon), pure vector similarity can rank a
paraphrase above the literal-keyword match. **Hybrid search**
(`query_type="hybrid"`) combines vector scoring with BM25-style
keyword scoring and boosts chunks that contain the literal
phrase. This is the exam-relevant alternative to "increase top-k
and rerank" for phrase-presence problems.

Source: https://docs.databricks.com/aws/en/vector-search/query-vector-search
The canonical doc example uses the lowercase value `query_type="hybrid"`
(alongside `"FULL_TEXT"` for full-text and the default `"ann"` for
vector-only). The kwarg name itself has been stable across recent
`databricks-vectorsearch` releases; the try/except below is a thin
safety net for the rare older build that lacked the keyword.

In [0]:
try:
    hybrid = index.similarity_search(
        query_text="chronic conditions",
        columns=["chunk_id", "doc_id", "tier", "state", "section"],
        num_results=4,
        filters={"state": "TX", "tier": "PawPlus"},
        query_type="hybrid",
    )
    print("Top-4 hybrid (vector + keyword), filtered (state=TX, tier=PawPlus):")
    display(pd.DataFrame(
        hybrid["result"]["data_array"],
        columns=[c["name"] for c in hybrid["manifest"]["columns"]],
    ))
except TypeError as e:
    # Older SDK versions may not expose query_type as a kwarg.
    print(f"Hybrid not available in this SDK build: {e}")

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
Top-4 hybrid (vector + keyword), filtered (state=TX, tier=PawPlus):


chunk_id,doc_id,tier,state,section,score
pp_plus_tx_v3__chunk_018,pp_plus_tx_v3,PawPlus,TX,4.7 Chronic condition exclusions and reimbursement handling,1.0
pp_plus_tx_v3__chunk_000,pp_plus_tx_v3,PawPlus,TX,front-matter,0.9684979838709677
pp_plus_tx_v3__chunk_022,pp_plus_tx_v3,PawPlus,TX,Section 6 Additional terms and conditions,0.9606894841269842
pp_plus_tx_v3__chunk_013,pp_plus_tx_v3,PawPlus,TX,4.2 Pre-existing condition exclusion,0.9471593644679827


## Recall@4 on Sarah's question — the production gate

**Recall@4 ≥ 0.8** is the PolicyPal production
ship gate. The number is computed against a `policy_qa_ground_truth`
table that pairs each question with its `expected_chunk_id`.
Below is the one-question version of that measurement using Sarah's
seed query; the c1301 evaluation notebook scales it to the full
`mlflow.genai.evaluate(...)` harness with `RetrievalSufficiency`
and a custom citation scorer across the seed eval set.

In [0]:
# Sarah's seed question + the chunk that should land in the top-4.
sarah_question = "Why was my reimbursement only $512 on an $890 bill?"
expected_chunk_id = "pp_plus_tx_v3__chunk_018"   # Section 4.7, deductible-handling

top4 = index.similarity_search(
    query_text=sarah_question,
    columns=["chunk_id"],
    num_results=4,
    filters={"state": "TX", "tier": "PawPlus"},
)
returned = [row[0] for row in top4["result"]["data_array"]]
hit = expected_chunk_id in returned
print(f"Returned top-4: {returned}")
print(f"Recall@4 for this single question: {1.0 if hit else 0.0}")
print(
    "Aggregate Recall@4 across the full eval set is computed by the c1301"
    " mlflow.genai.evaluate harness; production gate is >= 0.8."
)

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
Returned top-4: ['pp_plus_tx_v3__chunk_006', 'pp_plus_tx_v3__chunk_001', 'pp_plus_tx_v3__chunk_004', 'pp_plus_tx_v3__chunk_018']
Recall@4 for this single question: 1.0
Aggregate Recall@4 across the full eval set is computed by the c1301 mlflow.genai.evaluate harness; production gate is >= 0.8.


## What's next

* The ClaimClerk extraction chain (c0701) follows the chain-registration
  pattern; PolicyPal's RAG chain (c1001) calls this index
  with the customer's `state`/`tier` filters (read from the `policies`
  table by `customer_id`).
* The deployment notebook c1001 serves the chain on Model Serving with the index attached.
* The Delta Sync index auto-syncs when `policy_chunks` changes — e.g. after re-running the extraction + chunking notebooks for a new policy version.

## Teardown — delete the index 

**IMPORTANT to save money**

Removes the Vector Search index this notebook built.
Useful when iterating on chunk size / embedding model / metadata
columns — deleting and re-creating is faster than waiting for a
Delta Sync to re-embed an existing index with a different schema.
The endpoint itself is preserved; endpoint provisioning is a
workspace-admin action and is not destroyed from a chapter notebook.

The cell below is gated on `DELETE_INDEX = False` so a casual
"Run all" cannot accidentally remove the index. Flip the flag to
`True` and re-run just this cell to perform the deletion.

In [0]:
# ──────────────────────────────────────────────────────────────────────
# Gate flag — visible at the top. Flip to True to delete the index
# (preserves the VS endpoint itself; only the index goes).
#
# Source: https://api-docs.databricks.com/python/vector-search/latest/
# `VectorSearchClient.delete_index(index_name=...)` is the documented
# programmatic teardown — equivalent to REST
# `DELETE /api/2.0/vector-search/indexes/{index_name}`.
# ──────────────────────────────────────────────────────────────────────
DELETE_INDEX = False

In [0]:
if not DELETE_INDEX:
    print(
        f"⏸  Index teardown is OFF (DELETE_INDEX = False).\n"
        f"   To delete '{index_name}' from VS endpoint '{vs_endpoint}',\n"
        f"   set  DELETE_INDEX = True  at the top of the previous cell and re-run.\n"
        f"   The VS endpoint itself is preserved; only the index goes.\n"
        f"   ⚠  the agent, deployment, and eval notebooks all read from this index — only\n"
        f"      delete after finishing the book or when iterating on\n"
        f"      chunk size / embedding model / metadata columns."
    )
else:
    try:
        listed = vsc.list_indexes(name=vs_endpoint)
    except TypeError:
        listed = vsc.list_indexes(endpoint_name=vs_endpoint)
    existing = [i.get("name") for i in listed.get("vector_indexes", [])]
    if index_name not in existing:
        print(f"Index {index_name} does not exist on {vs_endpoint}; nothing to delete.")
    else:
        vsc.delete_index(index_name=index_name)
        print(f"Deleted index: {index_name} (endpoint {vs_endpoint} preserved).")

**Now remove the point. IMPORTANT to save money!**

![image_1780686686558.png](./image_1780686686558.png "image_1780686686558.png")